# sneaker-damage-seg: YOLOv8-seg 학습

준비: Google Drive의 `MyDrive/sneaker-damage-seg/` 폴더에 `sneaker_yolo2048.zip`을 올려 둡니다.
런타임 → 런타임 유형 변경 → **T4 GPU**를 고른 뒤 위에서부터 차례로 실행합니다.

규칙: 모델 선택은 val로만 하고, test는 실험마다 마지막에 한 번만 평가합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/sneaker-damage-seg'
RUNS = f'{DRIVE}/runs'          # 학습 결과는 Drive에 저장 (런타임이 끊겨도 남음)
!mkdir -p {RUNS}
!rm -rf /content/ds && mkdir -p /content/ds
!unzip -q {DRIVE}/sneaker_yolo2048.zip -d /content/ds   # 로컬 디스크가 Drive보다 훨씬 빠름
%cd /content/ds
!sed -i 's#^path: .*#path: /content/ds/yolo#' yolo/data.yaml
!cat yolo/data.yaml && ls yolo/images/*/ | head -3

In [ ]:
!pip install -q ultralytics
import ultralytics; ultralytics.checks()

## 실험 1: base_640 (기준 모델)
T4에서 50 epoch 약 30~60분 걸립니다.

In [ ]:
!yolo segment train model=yolov8n-seg.pt data=yolo/data.yaml imgsz=640 epochs=50 batch=16 seed=0 deterministic=True \
    project={RUNS} name=base_640 exist_ok=True

In [ ]:
# test 평가 (클래스별 Box/Mask P, R, mAP50, mAP50-95)
!yolo segment val model={RUNS}/base_640/weights/best.pt data=yolo/data.yaml split=test imgsz=640 \
    project={RUNS} name=base_640_test exist_ok=True

# 오탐·누락 분석용 예측 저장 후 사진별 리포트
!yolo segment predict model={RUNS}/base_640/weights/best.pt source=yolo/images/test imgsz=640 \
    save=False save_txt=True save_conf=True project={RUNS} name=base_640_pred exist_ok=True
!python tools/eval_report.py --labels yolo/labels/test --preds {RUNS}/base_640_pred/labels --out {RUNS}/base_640_report

## 실험 2: size_1024 (작은 손상 대응)
손상의 74%가 이미지 면적 0.1% 미만이라 입력을 키우는 효과를 봅니다. 나머지 조건은 실험 1과 같습니다. 약 1~2시간.

In [ ]:
!yolo segment train model=yolov8n-seg.pt data=yolo/data.yaml imgsz=1024 epochs=50 batch=8 seed=0 deterministic=True \
    project={RUNS} name=size_1024 exist_ok=True
!yolo segment val model={RUNS}/size_1024/weights/best.pt data=yolo/data.yaml split=test imgsz=1024 \
    project={RUNS} name=size_1024_test exist_ok=True
!yolo segment predict model={RUNS}/size_1024/weights/best.pt source=yolo/images/test imgsz=1024 \
    save=False save_txt=True save_conf=True project={RUNS} name=size_1024_pred exist_ok=True
!python tools/eval_report.py --labels yolo/labels/test --preds {RUNS}/size_1024_pred/labels --out {RUNS}/size_1024_report

## 결과 모아 보기
아래 셀의 출력을 그대로 복사해서 전달하면 실험 기록에 정리합니다.

In [ ]:
import pandas as pd, pathlib
for name in ['base_640', 'size_1024']:
    csv = pathlib.Path(RUNS) / name / 'results.csv'
    if not csv.exists():
        continue
    df = pd.read_csv(csv); df.columns = [c.strip() for c in df.columns]
    best = df.loc[df['metrics/mAP50(M)'].idxmax()]
    print(f"== {name}: best val epoch {int(best['epoch'])}  "
          f"mask mAP50 {best['metrics/mAP50(M)']:.3f}  mask mAP50-95 {best['metrics/mAP50-95(M)']:.3f}  "
          f"box mAP50 {best['metrics/mAP50(B)']:.3f}")
    rep = pathlib.Path(RUNS) / f'{name}_report' / 'summary.txt'
    if rep.exists():
        print(rep.read_text())